# Практика: фильтры и типы

Тот же `trips.csv`. Фильтры — первый инструмент аналитика: «покажи только центр», «только длинные поездки».

In [ ]:
from pathlib import Path
import pandas as pd


def find_trips_csv() -> Path:
    for p in (Path('trips.csv'), Path('../../data/trips.csv'), Path('../data/trips.csv')):
        if p.exists():
            return p.resolve()
    raise FileNotFoundError('trips.csv не найден')


TRIPS_PATH = find_trips_csv()
df = pd.read_csv(TRIPS_PATH)


## 1. Фильтр по зоне

Оставьте строки с `zone == 'center'`. Сколько таких поездок?

**Why:** маска — Series из True/False той же длины, что `df`.

**Вопрос:** строка в `center` — подмножество таблицы или новый файл?

In [ ]:
center = None  # df[маска]
assert center is not None
assert (center['zone'] == 'center').all()
assert len(center) > 0
print('center:', len(center), 'из', len(df))

## 2. Длинные поездки

`distance_km >= 15`. Выведите `trip_id`, расстояние, длительность (`head`).

**Pitfall:** сравнивайте с числом (`15`), не со строкой `'15'`.

In [ ]:
long_trips = None
assert long_trips is not None
assert (long_trips['distance_km'] >= 15).all()
print(long_trips[['trip_id', 'distance_km', 'duration_min']].head())

## 3. Составная маска (&)

Нужны поездки **и** в center, **и** с `distance_km >= 10`. Так **не работает**: `df['zone'] == 'center' and df['distance_km'] >= 10` — Python сравнивает два Series через `and` и падает.

**How:** `&` и скобки вокруг каждого условия: `df[(...) & (...)]`.

In [ ]:
center_long = None  # df[(...) & (...)]
assert center_long is not None
assert (center_long['zone'] == 'center').all()
assert (center_long['distance_km'] >= 10).all()
print(len(center_long))

## 4. Отрицание ~

Оставьте поездки **не** из `center` (`zone != 'center'` или `~ (df['zone'] == 'center')`).

**Checkpoint:** `len(not_center) + len(center)` должно равняться `len(df)` (если нет NaN в zone).

In [ ]:
not_center = None
assert not_center is not None
assert (not_center['zone'] != 'center').all()
assert len(not_center) + len(center) == len(df)
print('not center:', len(not_center))

## 5. Сортировка и топ-N

Три самые длинные поездки по `duration_min` — только `trip_id` и `duration_min`.

**How:** `sort_values(..., ascending=False).head(3)`.

In [ ]:
top3 = None
assert top3 is not None
assert len(top3) == 3
assert top3['duration_min'].is_monotonic_decreasing
print(top3[['trip_id', 'duration_min']])

## 6. Доля фильтра

Доля поездок `zone == 'center'` среди всех (0–1, округлить до 3 знаков).

**Why:** абсолютное число строк без доли плохо сравнивать между выборками разного размера.

In [ ]:
share_center = None
assert share_center is not None
assert 0 < float(share_center) < 1
print('доля center:', share_center)

## 7. loc vs iloc после фильтра

После фильтра `center` индекс может быть «дырявым». Возьмите **первую строку** `center` через `.iloc[0]` и запишите её `trip_id`.

**Pitfall:** `center.loc[0]` часто падает — метки 0 может не быть.

In [ ]:
first_center_id = None  # str trip_id
assert first_center_id is not None
assert isinstance(first_center_id, str) and first_center_id.startswith('T')
print(first_center_id)

## 8. Тип hour

Приведите `hour` к `int`, проверьте диапазон 0–23. Час — **категория времени суток**, не дробное число для «средней температуры».

In [ ]:
hours = None  # df['hour'].astype(int)
assert hours is not None
assert hours.dtype.kind in 'iu'
assert hours.min() >= 0 and hours.max() <= 23
print(int(hours.min()), int(hours.max()))